# Archived training entry point
Imported on 26 October 2026 from Kaggle version 360434584. This is historical source, not a new training run. Outputs and original metadata were removed, and `RUN_GPU_TRAINING` is set to `False`. The original settings and source checks remain unchanged. This notebook requires the authorized original Kaggle input bundles. See ../README.md and ../provenance.json.


# Knee ConvNeXt-Tiny Budget
Internet **off**. Attach the completed ConvNeXt CPU preparation output, its V1 parent preparation and unchanged original inputs.
One explicitly enabled training attempt, capped at 240 minutes and eight epochs. The first 25 successful updates are retained; five warmup updates are excluded from the budget estimate. The full 637-study validation BCE selects the checkpoint. Expert 58 studies are evaluated once after selection, as development diagnostics. No automatic retry, resume, extension, download or submission.
This entry is **enabled** for the authorized saved GPU run. Return the editable draft to the safe entry and accelerator None after saving.


In [ ]:
from pathlib import Path
import json
import sys
import torch

RUN_GPU_TRAINING = False
MAX_GPU_MINUTES = 240
BUNDLE = Path("/kaggle/input/notebooks/owaiskhan15/knee-convnext-cpu-preparation/knee_convnext_preparation/bundle")
assert (BUNDLE / "preflight.json").is_file(), "Attach the successful ConvNeXt CPU preparation output"
sys.path.insert(0, str(BUNDLE))
import cnn_convnext as runtime
import convnext_train as runner

assert Path(runtime.__file__).resolve().parent == BUNDLE.resolve()
assert Path(runner.__file__).resolve().parent == BUNDLE.resolve()
root, cfg, frame, pixels = runtime.load_bundle(BUNDLE, require_preflight=True)
expected_sources = {'cnn_convnext.py': '2b48dbe18a32e4a5e9c8c9adb83250e0f399c49343ff5cda6b0155bd855f29cd', 'convnext_train.py': 'ac3516fee36c3a8b0b6191592e09844de52ab6803d41eda779ac4dbf912a6f3a', 'prepare_convnext.py': '9052cd372119aaad01cbb7e32a1d530a9f05d099ce6fc443be4d334f9101bb0b'}
for name, expected in expected_sources.items():
    assert cfg["hashes"][name] == expected
    assert runtime.sha256(BUNDLE / name) == expected
preflight = json.loads((BUNDLE / "preflight.json").read_text())
assert preflight["complete"] and preflight["gpu_used"] is False
assert preflight["data_fingerprint"] == runtime.fingerprint(cfg)
assert preflight["encoder_backward_verified"] and preflight["missing_mask_verified"]
assert cfg["max_gpu_minutes"] == MAX_GPU_MINUTES
assert frame.groupby("split").size().to_dict() == {"fit": 3655, "weak_validation": 637, "gold": 58}
assert frame.groupby("verified_group").split.nunique().max() == 1
settings = runner._settings(cfg)
print("Verified ConvNeXt-Tiny bundle:", runtime.fingerprint(cfg), flush=True)
print("Training / weak validation / expert development:", 3655, 637, 58, flush=True)
print("Settings:", settings, flush=True)
print("Automatic retry, resume, download and submission: disabled", flush=True)

if RUN_GPU_TRAINING:
    assert torch.cuda.is_available(), "Explicit enabled attempt requires a GPU"
    result = runner.run_training(BUNDLE, max_minutes=MAX_GPU_MINUTES)
else:
    assert not torch.cuda.is_available(), "Safe draft must use accelerator None"
    print("CPU gate verified. Training remains disabled; no GPU used.", flush=True)
